# Notebook Setup

## Setting Directories

In [5]:
import sys
import os
from pathlib import Path

# Path to your project root
project_dir = Path(r"C:\Users\dmika\DEV\Projects-local\dp100-learn")

# Change the working directory
os.chdir(project_dir)

# Add to sys.path if not already there
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

## Imports

### Local Imports

In [6]:
# Now you can import from utils
from utils.consts import SUBSCRIPTION_ID, PREFERED_RESOURCE_LOCATION, MAIN_STORAGE_ACCOUNT_ACCESS_KEY

### General Imports

In [7]:
import numpy as np
import pandas as pd

### Azure Imports

In [8]:
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient

## Consts

In [9]:
subscription_id = SUBSCRIPTION_ID
azure_credentials = DefaultAzureCredential()

## Other

In [10]:
resource_group_name = "ml-workspace-dev"
resource_group_location = PREFERED_RESOURCE_LOCATION

In [11]:
storage_account_name = "dmpdp100storageaccount99"  # must be globally unique
storage_account_location = PREFERED_RESOURCE_LOCATION
storage_container_name = "dmpdp100data"
storage_account_access_key = MAIN_STORAGE_ACCOUNT_ACCESS_KEY

In [12]:
azureml_workspace_name = "mlw-dp100-labs"
azureml_resource_location = PREFERED_RESOURCE_LOCATION

In [13]:
datastore_name = "dmdp100datastore"

In [14]:
ml_client = MLClient(
    credential=azure_credentials,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    workspace_name=azureml_workspace_name
)

# Setup a Resource Group

In [11]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.resource import ResourceManagementClient

# Create resource management client
resource_client = ResourceManagementClient(azure_credentials, subscription_id)
try:
    # Try to get existing resource group
    rg_result = resource_client.resource_groups.get(resource_group_name)
    print(f"Resource group '{rg_result.name}' already exists in region '{rg_result.location}'")
except ResourceNotFoundError:
    # Create resource group if it doesn't exist
    rg_result = resource_client.resource_groups.create_or_update(
        resource_group_name,
        {"location": resource_group_location}
    )
    print(f"Provisioned resource group '{rg_result.name}' in the {rg_result.location} region")


# Optional lines to delete the resource group. begin_delete is asynchronous.
# poller = resource_client.resource_groups.begin_delete(rg_result.name)
# result = poller.result()

Resource group 'ml-workspace-dev' already exists in region 'westeurope'


# Setup Storage Account

## Create a Storage Account

In [13]:
from azure.core.exceptions import ResourceNotFoundError
from azure.mgmt.storage import StorageManagementClient

storage_client = StorageManagementClient(azure_credentials, subscription_id)

try:
    # Try to get existing storage account
    storage_account = storage_client.storage_accounts.get_properties(
        resource_group_name=resource_group_name,
        account_name=storage_account_name
    )
    print(f"Storage account already exists: {storage_account.name}")
except ResourceNotFoundError:
    # If not found, create new storage account
    print("Creating storage account...")
    poller = storage_client.storage_accounts.begin_create(
        resource_group_name=resource_group_name,
        account_name=storage_account_name,
        parameters={
            "location": storage_account_location,
            "sku": {"name": "Standard_LRS"},
            "kind": "StorageV2",
            "enable_https_traffic_only": True
        }
    )
    storage_account = poller.result()
    print(f"Storage account created: {storage_account.name}")

# Extract the resource ID
storage_resource_id = storage_account.id
print(f"Storage Resource ID: {storage_resource_id}")

Storage account already exists: dmpdp100storageaccount99
Storage Resource ID: /subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.Storage/storageAccounts/dmpdp100storageaccount99


## Create Data Container

In [14]:
from azure.storage.blob import BlobServiceClient

# Build connection string
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};"
    f"AccountKey={storage_account_access_key};EndpointSuffix=core.windows.net"
)

# Create blob service client
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create container (if not exists)
try:
    blob_service_client.create_container(storage_container_name)
    print(f"Container '{storage_container_name}' created.")
except Exception as e:
    if "ContainerAlreadyExists" in str(e):
        print(f"Container '{storage_container_name}' already exists.")
    else:
        raise

Container 'dmpdp100data' created.


# Setup Azure ML Workspace

## Creating the Workspace

In [14]:
from azure.core.exceptions import ResourceNotFoundError
from azure.ai.ml.entities import Workspace

try:
    # Try to get existing workspace
    ws = ml_client.workspaces.get(azureml_workspace_name)
    print(f"AML workspace already exists: {ws.name}")
except (ResourceNotFoundError, TypeError):
    ml_client = MLClient(
        credential=azure_credentials,
        subscription_id=subscription_id,
        resource_group_name=resource_group_name,
    )
    # Create new AML workspace
    ws = Workspace(
        name=azureml_workspace_name,
        location=azureml_resource_location,
        storage_account=storage_resource_id
    )
    print("Creating AML workspace...")
    ml_client.workspaces.begin_create(ws).result()
    print(f"Workspace '{azureml_workspace_name}' created with default storage: {storage_account_name}")
    ml_client = MLClient(
        credential=azure_credentials,
        subscription_id=subscription_id,
        resource_group_name=resource_group_name,
        workspace_name=azureml_workspace_name
    )


AML workspace already exists: mlw-dp100-labs


## Setup the Data

### Create a DataStore

In [15]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration

store = AzureBlobDatastore(
    name=datastore_name,
    description="Blob Storage for DP-100 certification prep",
    account_name=storage_account_name,
    container_name=storage_container_name, 
    credentials=AccountKeyConfiguration(
        account_key=storage_account_access_key
    ),
    type="azure_blob",
)

ml_client.create_or_update(store)

AzureBlobDatastore({'type': <DatastoreType.AZURE_BLOB: 'AzureBlob'>, 'name': 'dmdp100datastore', 'description': 'Blob Storage for DP-100 certification prep', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/datastores/dmdp100datastore', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': None, 'serialize': <msrest.serialization.Serializer object at 0x000001EFC4F5FDF0>, 'credentials': {'type': 'account_key'}, 'container_name': 'dmpdp100data', 'account_name': 'dmpdp100storageaccount99', 'endpoint': 'core.windows.net', 'protocol': 'https'})

In [16]:
stores = ml_client.datastores.list()
for ds_name in stores:
    print(ds_name.name)

dmdp100datastore
workspacefilestore
workspaceblobstore
workspaceworkingdirectory
workspaceartifactstore


### Create Data Assets

In [19]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

file_path = './data/telco-churn-data/telco-customer-churn.csv'

my_data = Data(
    path=file_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FILE,
    description="Data asset pointing to a local file, automatically uploaded to the default datastore",
    name="telco-churn-file-raw"
)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/c9f74bf3dd2417bba280f0ceef276024cbe95a3935ed06a94a7f357d15495775/telco-customer-churn.csv', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_file', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-file-raw', 'description': 'Data asset pointing to a local file, automatically uploaded to the default datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-file-raw/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.Syst

In [20]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

folder_path = './data/telco-churn-data'

my_data = Data(
    path=folder_path,
    datastore=datastore_name,
    type=AssetTypes.URI_FOLDER,
    description="Data asset pointing to data-asset-path folder in datastore",
    name="telco-churn-folder-raw",
)

ml_client.data.create_or_update(my_data)

Uploading telco-churn-data (0.97 MBs): 100%|##########| 970595/970595 [00:00<00:00, 2886586.66it/s]




Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/67bbe0434ee4e9154f85f403f18de4e49ef43c66b5844f85418a3218b89d8b33/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': None, 'type': 'uri_folder', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-folder-raw', 'description': 'Data asset pointing to data-asset-path folder in datastore', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-folder-raw/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000001EF

In [21]:
%%writefile data/telco-churn-data/MLTable

paths:
  - file: ./telco-customer-churn.csv
transformations:
  - read_delimited:
        delimiter: ','
        encoding: 'ascii'

Overwriting data/telco-churn-data/MLTable


In [22]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data_path = './data/telco-churn-data'

my_data = Data(
    path=data_path,
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to telco-customer-churn.csv in data folder",
    name="telco-churn-table-raw",
    datastore=datastore_name,

)

ml_client.data.create_or_update(my_data)

Data({'path': 'azureml://subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs/datastores/dmdp100datastore/paths/LocalUpload/67bbe0434ee4e9154f85f403f18de4e49ef43c66b5844f85418a3218b89d8b33/telco-churn-data/', 'skip_validation': False, 'mltable_schema_url': None, 'referenced_uris': ['./telco-customer-churn.csv'], 'type': 'mltable', 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'telco-churn-table-raw', 'description': 'MLTable pointing to telco-customer-churn.csv in data folder', 'tags': {}, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/data/telco-churn-table-raw/versions/1', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemDat

### Read the Data Assets Locally

In [25]:
data_asset = ml_client.data.get("telco-churn-file-raw", version="1")
df = pd.read_csv(data_asset.path)
df.sample(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
5159,5928-QLDHB,Male,0,No,No,9,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,76.25,684.85,No
2645,8562-GHPPI,Female,0,No,No,1,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Credit card (automatic),19.80,19.8,No


In [26]:
# import mltable

data_asset = ml_client.data.get("telco-churn-folder-raw", version="1")
path = {
  'folder': data_asset.path
}
# tbl = mltable.from_delimited_files(paths=[path])
# df = tbl.to_pandas_dataframe()
df = pd.read_csv(os.path.join(data_asset.path, 'telco-customer-churn.csv'))
df.sample(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4896,3521-MNKLV,Male,0,No,No,24,Yes,No,DSL,No,...,Yes,No,No,No,Month-to-month,Yes,Mailed check,49.7,1167.8,No
5884,4785-QRJHC,Male,1,Yes,No,46,No,No phone service,DSL,No,...,Yes,Yes,Yes,Yes,One year,Yes,Bank transfer (automatic),59.9,2816.65,Yes


In [28]:
import mltable

data_asset = ml_client.data.get("telco-churn-table-raw", version="1")

tbl = mltable.load(f"azureml:/{data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,False,True,False,1,False,No phone service,DSL,No,...,No,No,No,No,Month-to-month,True,Electronic check,29.85,29.85,False
1,5575-GNVDE,Male,False,False,False,34,True,No,DSL,Yes,...,Yes,No,No,No,One year,False,Mailed check,56.95,1889.50,False
2,3668-QPYBK,Male,False,False,False,2,True,No,DSL,Yes,...,No,No,No,No,Month-to-month,True,Mailed check,53.85,108.15,True
3,7795-CFOCW,Male,False,False,False,45,False,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,False,Bank transfer (automatic),42.30,1840.75,False
4,9237-HQITU,Female,False,False,False,2,True,No,Fiber optic,No,...,No,No,No,No,Month-to-month,True,Electronic check,70.70,151.65,True


## Setup an Environment

In [15]:
env_file_path = "python_env_setup/dmdp100env.yml"

In [ ]:
# %%writefile $env_file_path
# name: dmdp100env
# channels:
#   - conda-forge
# dependencies:
#   - python=3.11
#   - scikit-learn
#   - pandas
#   - numpy
#   - matplotlib
#   - pip
#   - pip:
#     - seaborn
#     - mltable
#     - mlflow
#     - azureml-mlflow

Overwriting python_env_setup/dmdp100env.yml


In [21]:
%%writefile $env_file_path
name: dmdp100env
channels:
  - conda-forge
dependencies:
  - python=3.11
  - scikit-learn
  - pandas
  - numpy
  - matplotlib
  - pip
  - pip:
    - seaborn

Overwriting python_env_setup/dmdp100env.yml


In [23]:
from azure.ai.ml.entities import Environment

env_docker_conda = Environment(
    image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
    conda_file=env_file_path,
    name="dmpdp100env",
    description="Environment created for main dp100 prep.",
)
ml_client.environments.create_or_update(env_docker_conda)

Environment({'arm_type': 'environment_version', 'latest_version': None, 'image': 'mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04', 'intellectual_property': None, 'is_anonymous': False, 'auto_increment_version': False, 'auto_delete_setting': None, 'name': 'dmpdp100env', 'description': 'Environment created for main dp100 prep.', 'tags': {}, 'properties': {'azureml.labels': 'latest'}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/environments/dmpdp100env/versions/4', 'Resource__source_path': '', 'base_path': 'C:\\Users\\dmika\\DEV\\Projects-local\\dp100-learn', 'creation_context': <azure.ai.ml.entities._system_data.SystemData object at 0x000002F34CA671C0>, 'serialize': <msrest.serialization.Serializer object at 0x000002F34CA67AC0>, 'version': '4', 'conda_file': {'channels': ['conda-forge'], 'dependencies': ['python=3.11', 'scikit-learn', 'pandas', 

## Setup the Compute

### Compute Cluster

In [37]:
from azure.ai.ml.entities import AmlCompute

# Name assigned to the compute cluster
cpu_compute_target = "dmdp100-cpu-cluster"

try:
    # let's see if the compute target already exists
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(
        f"You already have a cluster named {cpu_compute_target}, we'll reuse it as is."
    )

except Exception:
    print("Creating a new cpu compute target...")

    # Let's create the Azure ML compute object with the intended parameters
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        # Azure ML Compute is the on-demand VM service
        type="amlcompute",
        # VM Family
        size="STANDARD_DS11_V2",
        # Minimum running nodes when there is no job running
        min_instances=0,
        # Nodes in cluster
        max_instances=1,
        # How many seconds will the node running after the job termination
        idle_time_before_scale_down=120,
        # Dedicated or LowPriority. The latter is cheaper but there is a chance of job termination
        tier="Dedicated",
    )

    # Now, we pass the object to MLClient's create_or_update method
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster)


Creating a new cpu compute target...


### Compute Instance

In [35]:
# Compute Instances need to have a unique name across the region.
# Here we create a unique name with current datetime
from azure.ai.ml.entities import ComputeInstance
import datetime

ci_basic_name = "dp100ci" + datetime.datetime.now().strftime("%Y%m%d%H%M")
ci_basic_name = ci_basic_name[:24]
ci_basic = ComputeInstance(name=ci_basic_name, size="STANDARD_DS11_V2", idle_time_before_shutdown_minutes="15")
ml_client.begin_create_or_update(ci_basic).result()

ComputeInstance({'state': 'Running', 'last_operation': {'operation_name': 'Create', 'operation_time': '2025-09-10T17:21:46.731Z', 'operation_status': 'Succeeded', 'operation_trigger': 'User'}, 'os_image_metadata': <azure.ai.ml.entities._compute._image_metadata.ImageMetadata object at 0x00000257F19429C0>, 'services': [{'display_name': 'Jupyter', 'endpoint_uri': 'https://dp100ci202509101921.westeurope.instances.azureml.ms/tree/'}, {'display_name': 'Jupyter Lab', 'endpoint_uri': 'https://dp100ci202509101921.westeurope.instances.azureml.ms/lab'}], 'type': 'computeinstance', 'created_on': '2025-09-10T17:21:33.372399+0000', 'provisioning_state': 'Succeeded', 'provisioning_errors': None, 'name': 'dp100ci202509101921', 'description': None, 'tags': None, 'properties': {}, 'print_as_yaml': False, 'id': '/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourceGroups/ml-workspace-dev/providers/Microsoft.MachineLearningServices/workspaces/mlw-dp100-labs/computes/dp100ci202509101921', 'Resource_